# Data Cleaning Pipeline — Demo

This notebook runs the full pipeline (**load → detect → clean → HTML report**) using `src` modules.

**Where to run:** From the **project root** or from `notebooks/` — the first code cell sets `ROOT` to the repository root so imports and the saved report path stay correct.

**Demo data:** `data/demo_sales/sales_messy.csv` — mixed dates, missing `amount`, duplicate keys, inconsistent `region`. Issue detection also includes **IQR outliers** (report-only; not auto-removed).

In [ ]:
import sys
from pathlib import Path

# Repository root (works if cwd is project root or notebooks/)
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.load import load_demo_sales
from src.detect import detect_issues
from src.clean import basic_clean
from src.report import build_summary_table, save_html_report, report_to_bytes

## 1. Load data

In [ ]:
df = load_demo_sales()
print(f"Rows: {len(df)}, Columns: {list(df.columns)}")
df

## 2. Detect issues

In [ ]:
issues = detect_issues(df, missing_threshold=0.5)
print("Duplicate rows:", issues["duplicates"]["duplicate_row_count"])
print("Invalid formats:", issues["invalid_formats"])
print("Inconsistent categories:", issues["inconsistent_categories"])
print("Outliers (IQR):", issues.get("outliers", {}))
print("Missing (per column):", [c for c in issues["missing"]["per_column"] if c["missing_ratio"] > 0])

## 3. Clean

In [ ]:
cleaned, actions = basic_clean(df)
print(f"Rows before: {len(df)} → after: {len(cleaned)}")
for a in actions:
    print(" -", a.description)
cleaned.head()

## 4. Report

In [ ]:
summary = build_summary_table(df, cleaned, issues, actions)
assert "outliers" in summary and "schema_comparison" in summary

out_path = ROOT / "artifacts" / "reports" / "notebook_demo_report.html"
save_html_report(summary, out_path)
print(f"Report saved: {out_path.resolve()}")

# Same HTML bytes as the Streamlit app uses for download
html_bytes = report_to_bytes(summary)
print(f"Report HTML size (bytes): {len(html_bytes)}")